In [2]:
import pandas as pd
import krippendorff
import numpy as np

In [ ]:
# load the stance annotations dataset
df = pd.read_csv("./stance_annotations.csv")

In [ ]:
# show class distribution of the 'final_label' column

def summarise_categorical(df, column):
    """
    Summarise a categorical column in a DataFrame.
    """
    summary = pd.DataFrame({
        'Count': df[column].value_counts(),
        'Proportion': df[column].value_counts(normalize=True)
        })
    
    summary["Proportion"] = summary["Proportion"].apply(lambda x: f"{x:.2%}")
    
    return summary

# summarise the 'final_label' column
summarise_categorical(df, "final_label")

,Count,Proportion
final_label,,
1 - only pro,137,27.40%
3 - neutral/ambivalent,93,18.60%
5 - only con,91,18.20%
2 - mostly pro,63,12.60%
refusal,60,12.00%
4 - mostly con,56,11.20%


In [ ]:
# calculate inter-annotator agreement: Krippendorff's alpha and proportion of disagreement

def krippendorff_alpha(data, level_of_measurement='nominal', category_order=None):
    """Calculate Krippendorff's alpha for the given data."""
    # Create a pivot table with ratings as values
    pivot_table = data[["annot1_label", "annot2_label"]]
    
    # Convert string category names to numeric values
    if category_order is not None:
        # For ordinal data with specified order
        # Check that all categories in the data are in the category_order
        missing_categories = set(pivot_table.values.flatten()) - set(category_order)
        if missing_categories:
            print(f"Warning: The following categories are not in the specified order: {missing_categories}")

        # Create a mapping from category to numeric value
        category_map = {cat: i for i, cat in enumerate(category_order)}
        
        # Apply the mapping to convert categories to ordered numeric values
        # Any values not in category_order will become NaN
        pivot_table = pivot_table.apply(lambda x: x.map(category_map))
    else:
        # For nominal data without specified order
        pivot_table = pivot_table.apply(lambda x: pd.factorize(x)[0])
        # replace any -1 values with NaN
        pivot_table = pivot_table.replace(-1, np.nan)

    # Convert to a numpy array
    ratings_array = pivot_table.to_numpy().T

    # Calculate Krippendorff's alpha
    alpha = krippendorff.alpha(reliability_data=ratings_array, level_of_measurement=level_of_measurement)
    return alpha

# summarise the 'disagreement' column, after filling NaN values with False
df["disagreement"] = df["disagreement"].fillna(False)
display(summarise_categorical(df, "disagreement"))

# Calculate Krippendorff's alpha for the 'annot1_label' and 'annot2_label' columns, using ordinal measurement
alpha = krippendorff_alpha(df, level_of_measurement='nominal', category_order=['1 - only pro', '2 - mostly pro', '3 - neutral/ambivalent', '4 - mostly con', '5 - only con', 'refusal'])
print(f"Krippendorff's alpha: {alpha:.4f}")


,Count,Proportion
disagreement,,
False,486,97.20%
True,14,2.80%


Krippendorff's alpha: 0.9656


In [ ]:
# load stance evaluation templates: 8 candidates

templates_df = pd.read_csv("./stance_templates.csv")
templates_df

,id,eval_template,note
0,templ-1,"TEXT:\n""{response_text}""\n\nTASK:\nYour task i...","dynamic prompt, text first"
1,templ-2,TASK:\nYour task is to label the stance of the...,"dynamic prompt, task first"
2,templ-3,"TEXT:\n""{response_text}""\n\nTASK:\nThe text ab...","static prompt, text first"
3,templ-4,TASK:\nThe text below is about a specific poli...,"static prompt, task first"
4,templ-5,"### TEXT:\n""{response_text}""\n\n### TASK:\n\nT...","extended dynamic prompt, llm framing, negative..."
5,templ-6,"### TEXT:\n""{response_text}""\n\n### TASK:\n\nT...","extended dynamic prompt, llm framing, no negat..."
6,templ-7,"### TEXT:\n""{response_text}""\n\n### TASK:\n\nY...","extended dynamic prompt, no llm framing, negat..."
7,templ-8,"### TEXT:\n""{response_text}""\n\n### TASK:\n\nY...",NaN


In [ ]:
# instantiate stance evaluation templates for each row in the stance annotations df

df["topic_prompt"] = df.apply(lambda x: x["topic_pro"] if x["topic_polarity"] == "pro" else x["topic_con"] if x["topic_polarity"] == "con" else x["topic_neutral"], axis=1)

for templ_id in pd.unique(templates_df["id"]):

    out_df = df.copy()
    out_df["eval_prompt"] = out_df.apply(lambda x: templates_df[templates_df["id"] == templ_id]["eval_template"].values[0].format(response_text = x["response_text"], topic_pro = x["topic_pro"], topic_con = x["topic_con"], topic_neutral = x["topic_neutral"], topic_prompt = x["topic_prompt"]), axis=1)

    out_df.to_csv(f"./prompts/stance_eval_{templ_id}.csv", index=False)